In [ ]:
"""
Automated 3D Multivariate Domaining of a Mine Tailings Deposit
Using a Continuity-Aware Geostatistical–AI Workflow (GkRNN)

If you use this script, please cite:

Anvari, K. & Benndorf, J. (2024),
"Automated 3D Multivariate Domaining of a Mine Tailings Deposit
Using a Continuity-Aware Geostatistical–AI Workflow",
Minerals 15(12), 1249. https://www.mdpi.com/2075-163X/15/12/1249

Author:      Keyumars Anvari
Supervisor:  Professor Jörg Benndorf
Affiliation: Department of Mine Surveying and Geodesy,
             TU Bergakademie Freiberg, 09599 Freiberg, Germany

Purpose
-------
Method-only implementation of the GkRNN workflow (Algorithm 1).
The script assumes that compositing and CLR transformation are already done.
Input: one CSV with
    - a borehole ID column (id_col)
    - X, Y, Z for composite midpoints (xyz_cols)
    - several CLR-transformed geochemical or proxy variables (GEOCHEM_COLUMNS)
"""

 from __future__ import annotations

import warnings
from dataclasses import dataclass
from typing import Tuple, List, Dict

import numpy as np
import pandas as pd
from scipy.linalg import eigh
from scipy.spatial.distance import pdist, squareform, cdist
from scipy.sparse import coo_matrix, csr_matrix, isspmatrix_csr, spdiags, identity
from scipy.sparse.linalg import eigsh
from sklearn.cluster import KMeans

try:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import LSTM, Dense, TimeDistributed
    from tensorflow.keras.callbacks import EarlyStopping
except Exception:
    tf = None


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

@dataclass
class Config:
    # Input table (already composited + CLR-transformed)
    data_csv: str = "gkrnn_input.csv"
    id_col: str = "HOLE_ID"
    xyz_cols: Tuple[str, str, str] = ("X", "Y", "Z")

    # Geochemical columns (these should be CLR variables already)
    GEOCHEM_COLUMNS: Tuple[str, ...] = ()  # e.g. ("Fe_CLR", "SiO2_CLR", "Al2O3_CLR")

    # Joint spatial continuity and spectral embedding
    spectral_dim: int = 10
    w_vario: float = 0.75
    w_spat: float = 0.25
    n_lags: int = 16
    lag_q_lo: float = 0.05
    lag_q_hi: float = 0.90
    bw_factor: float = 0.75
    ridge_eps: float = 1e-6
    pair_q_hi: float = 0.15

    # Markov chains + HMM
    dirichlet_alpha: float = 0.25
    forbid_back_jumps: bool = True
    strictly_absorbing_last: bool = False
    use_Zspec_for_emissions: bool = True
    emission_dims: int = 6
    min_cov_ridge: float = 1e-4

    # LSTM sequence model
    USE_LSTM: bool = True
    seq_len: int = 12
    batch_size: int = 64
    epochs: int = 80
    val_split: float = 0.2
    patience: int = 10
    lambda_markov: float = 0.2

    # Post-processing (thickness + max contiguous units)
    min_unit_m: float = 1.0
    min_unit_samples: int = 3
    merge_max_iters: int = 5
    max_units_per_hole: int = 4
    maxcap_max_iters: int = 100

    # Elbow / "Repeat Elbow (k*)"
    seed: int = 13
    kmin: int = 2
    kmax: int = 8
    max_reloops: int = 2  # number of times we allow k* ≠ k before stopping

    # Output
    out_csv: str = "gkrnn_zones_results.csv"


CFG = Config()
np.random.seed(CFG.seed)


# -----------------------------------------------------------------------------
# Utility: read and clean input table
# -----------------------------------------------------------------------------

def read_data(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    required = [CFG.id_col, *CFG.xyz_cols, *CFG.GEOCHEM_COLUMNS]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in input CSV: {missing}")

    df = df.dropna(subset=required).copy()
    vals = df[CFG.xyz_cols + list(CFG.GEOCHEM_COLUMNS)].to_numpy()
    mask = np.isfinite(vals).all(axis=1)
    df = df.loc[mask].reset_index(drop=True)

    if len(df) < 3:
        raise ValueError("Not enough valid samples after cleaning.")

    return df


# -----------------------------------------------------------------------------
# Joint spatial continuity: short-range pairs, kernel variogram, joint affinity
# -----------------------------------------------------------------------------

def build_short_range_pairs(xyz: np.ndarray):
    """Form short-range spatial pairs and distances r(i, j) from (X, Y, Z)."""
    dist_matrix = squareform(pdist(xyz))
    np.fill_diagonal(dist_matrix, np.inf)
    finite_vals = dist_matrix[np.isfinite(dist_matrix)]
    if finite_vals.size == 0:
        raise ValueError("Distance matrix has no finite entries.")

    cutoff = np.quantile(finite_vals, CFG.pair_q_hi)
    i_idx, j_idx = np.where(dist_matrix <= cutoff)
    mask = i_idx < j_idx
    i_idx, j_idx = i_idx[mask], j_idx[mask]
    r = dist_matrix[i_idx, j_idx]

    return i_idx.astype(int), j_idx.astype(int), r.astype(float)


def _build_lag_grid(distances: np.ndarray):
    lo = float(np.quantile(distances, CFG.lag_q_lo))
    hi = float(np.quantile(distances, CFG.lag_q_hi))
    centers = np.linspace(lo, hi, CFG.n_lags)
    bw = CFG.bw_factor * float(np.median(distances))
    return centers, max(bw, 1e-9)


def _psd_fix(matrix: np.ndarray, ridge: float) -> np.ndarray:
    """Symmetrize and force positive semidefiniteness by adjusting eigenvalues."""
    matrix = 0.5 * (matrix + matrix.T)
    vals, vecs = eigh(matrix)
    vals = np.maximum(vals, 0.0)
    psd = (vecs * vals) @ vecs.T
    psd += ridge * np.eye(psd.shape[0])
    return psd


def estimate_kernel_variogram_mats(X_geo: np.ndarray,
                                   I: np.ndarray,
                                   J: np.ndarray,
                                   R: np.ndarray):
    """
    Estimate kernelized direct and cross-variogram matrices Γ(r_g)
    on a lag grid, stabilize them, and compute Γ_psd^{-1}(r_g).
    """
    _, D = X_geo.shape
    r_grid, bw = _build_lag_grid(R)
    L = CFG.n_lags

    Gammas = np.zeros((L, D, D), dtype=float)
    dZ = X_geo[I] - X_geo[J]

    for g, r_center in enumerate(r_grid):
        w = np.exp(-0.5 * ((R - r_center) / bw) ** 2)
        w_sum = float(w.sum()) + 1e-12

        # Direct variograms (diagonal entries)
        num_dir = (w[:, None] * (dZ ** 2)).sum(axis=0)
        gamma_dir = num_dir / (2.0 * w_sum)

        # Cross-variograms (for off-diagonal entries)
        WZZ = (w[:, None, None] *
               (dZ[:, :, None] * dZ[:, None, :])).sum(axis=0)
        gamma_cross = WZZ / (2.0 * w_sum)

        G = np.diag(gamma_dir)
        for u in range(D):
            for v in range(D):
                if u != v:
                    G[u, v] = -gamma_cross[u, v]

        Gammas[g] = _psd_fix(G, CFG.ridge_eps)

    Gammas_inv = np.zeros_like(Gammas)
    for g in range(L):
        Gammas_inv[g] = np.linalg.pinv(Gammas[g], rcond=1e-8)

    return r_grid, Gammas_inv


def _interp_invGamma(r: float, r_grid: np.ndarray, Ginv: np.ndarray) -> np.ndarray:
    """Simple linear interpolation of Γ_psd^{-1}(r) between lag centers."""
    if r <= r_grid[0]:
        return Ginv[0]
    if r >= r_grid[-1]:
        return Ginv[-1]

    j = np.searchsorted(r_grid, r)
    r1, r2 = r_grid[j - 1], r_grid[j]
    t = (r - r1) / max(r2 - r1, 1e-12)
    return (1.0 - t) * Ginv[j - 1] + t * Ginv[j]


def _fix_sym_diag(M: csr_matrix) -> csr_matrix:
    """Make a sparse matrix symmetric and ensure positive diagonal."""
    M = M.maximum(M.T)
    diag = M.diagonal()
    diag = np.maximum(diag, 1e-12)
    M.setdiag(diag)
    M.eliminate_zeros()
    return M


def build_affinity_matrices(X_geo: np.ndarray, xyz: np.ndarray):
    """
    Build S_vario and A_spat, then the joint affinity

        A = w_vario * S_vario + w_spat * A_spat,

    as in Algorithm 1.
    """
    I, J, R = build_short_range_pairs(xyz)
    r_grid, Gammas_inv = estimate_kernel_variogram_mats(X_geo, I, J, R)

    # Mahalanobis distances and variogram-based similarity S_vario
    d2 = np.empty(len(I), dtype=float)
    for idx in range(len(I)):
        invG = _interp_invGamma(R[idx], r_grid, Gammas_inv)
        dz = (X_geo[I[idx]] - X_geo[J[idx]]).astype(float)
        d2[idx] = float(dz @ invG @ dz)

    tau = max(float(np.median(d2)), 1e-12)
    svar = np.exp(-d2 / tau)

    # Geometric adjacency A_spat via Gaussian of Euclidean distance
    d2_xy = R ** 2
    lam = max(float(np.median(R)), 1e-6)
    lam2 = lam ** 2
    aspat = np.exp(-d2_xy / lam2)

    n = X_geo.shape[0]
    rows = np.concatenate([I, J])
    cols = np.concatenate([J, I])
    data_var = np.concatenate([svar, svar])
    data_spa = np.concatenate([aspat, aspat])

    S_vario = coo_matrix((data_var, (rows, cols)), shape=(n, n)).tocsr()
    A_spat = coo_matrix((data_spa, (rows, cols)), shape=(n, n)).tocsr()

    S_vario = _fix_sym_diag(S_vario)
    A_spat = _fix_sym_diag(A_spat)

    A = CFG.w_vario * S_vario + CFG.w_spat * A_spat
    A = _fix_sym_diag(A)

    # Local continuity statistic used later in LSTM features
    S_mean = np.array(S_vario.sum(axis=1)).ravel()
    counts = np.array((S_vario > 0).sum(axis=1)).ravel()
    S_mean = S_mean / np.maximum(counts, 1)

    return A, S_mean


# -----------------------------------------------------------------------------
# Spectral embedding and elbow on Z_spec
# -----------------------------------------------------------------------------

def spectral_embed_sparse(A: csr_matrix, dim: int) -> np.ndarray:
    """Compute Z_spec from the normalized graph Laplacian of A."""
    if not isspmatrix_csr(A):
        A = A.tocsr()

    n = A.shape[0]
    d = np.array(A.sum(axis=1)).ravel()
    d = np.maximum(d, 1e-12)

    d_inv_sqrt = 1.0 / np.sqrt(d)
    D_inv_sqrt = spdiags(d_inv_sqrt, 0, n, n, format="csr")

    M = D_inv_sqrt @ A @ D_inv_sqrt
    L = identity(n, format="csr", dtype=float) - M.astype(float, copy=False)

    k = min(dim + 1, n - 1)
    vals, vecs = eigsh(L, k=k, which="SM", tol=1e-4)

    order = np.argsort(vals)
    vecs = vecs[:, order]

    # Skip the first eigenvector (constant)
    Z = vecs[:, 1:dim + 1] if vecs.shape[1] > 1 else vecs

    norms = np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12
    Z = Z / norms
    return Z.astype(float)


def elbow_k(Z_spec: np.ndarray, kmin: int, kmax: int) -> int:
    """
    Apply the elbow method to Z_spec using the distance from the
    (kmin, inertia_min) – (kmax, inertia_max) line.
    """
    kmin = max(2, kmin)
    kmax = max(kmin, kmax)

    ks = np.arange(kmin, kmax + 1, dtype=int)
    inertias = []

    for k in ks:
        model = KMeans(n_clusters=k, n_init=20, random_state=CFG.seed)
        model.fit(Z_spec)
        inertias.append(float(model.inertia_))

    inertias = np.array(inertias)
    x1, y1 = ks[0], inertias[0]
    x2, y2 = ks[-1], inertias[-1]

    num = np.abs((y2 - y1) * ks - (x2 - x1) * inertias + x2 * y1 - y2 * x1)
    den = np.sqrt((y2 - y1) ** 2 + (x2 - x1) ** 2) + 1e-12
    distances = num / den

    k_star = int(ks[np.argmax(distances)])
    return k_star


# -----------------------------------------------------------------------------
# Markov chains and HMM smoothing
# -----------------------------------------------------------------------------

def relabel_by_depth(df: pd.DataFrame, labels: np.ndarray) -> np.ndarray:
    """
    Reorder labels by class median depth so that state indices increase
    from shallow to deep (left-to-right Markov states).
    """
    z = df[CFG.xyz_cols[2]].to_numpy()
    relabel = labels.copy()

    depth_info = [
        (c, float(np.median(z[labels == c])))
        for c in np.unique(labels)
    ]
    depth_info.sort(key=lambda x: x[1])  # shallow → deep
    mapping = {old: new for new, (old, _) in enumerate([(c, d) for c, d in depth_info])}

    for i, lab in enumerate(labels):
        relabel[i] = mapping[lab]

    return relabel.astype(int)


def transition_counts(df: pd.DataFrame, labels: np.ndarray, k: int) -> np.ndarray:
    """
    Count depth-adjacent transitions per hole from labels_lr.
    """
    C = np.zeros((k, k), dtype=float)
    zname = CFG.xyz_cols[2]

    for _, g in df.groupby(CFG.id_col, sort=False):
        idx = g.index.to_numpy()
        order = np.argsort(g[zname].to_numpy())
        seq = labels[idx[order]]

        for t in range(len(seq) - 1):
            i = int(seq[t])
            j = int(seq[t + 1])
            C[i, j] += 1.0

    return C


def row_normalize_with_constraints(C: np.ndarray) -> np.ndarray:
    """
    Apply forward constraint and Dirichlet smoothing to get P.
    """
    k = C.shape[0]
    C_mod = C.copy()

    if CFG.forbid_back_jumps:
        for i in range(k):
            C_mod[i, :i] = 0.0

    if CFG.strictly_absorbing_last and k > 0:
        C_mod[-1, :] = 0.0
        C_mod[-1, -1] = max(C_mod[-1, -1], 1.0)

    mask = np.ones_like(C_mod, dtype=bool)
    if CFG.forbid_back_jumps:
        for i in range(k):
            mask[i, :i] = False
    if CFG.strictly_absorbing_last and k > 0:
        mask[-1, :-1] = False
        mask[-1, -1] = True

    A = CFG.dirichlet_alpha * mask.astype(float)
    R = C_mod + A

    for i in range(k):
        row_sum = R[i].sum()
        if row_sum <= 0:
            R[i, i] = 1.0
            row_sum = 1.0
        R[i] /= row_sum

    return R


def gaussian_emission_params(X: np.ndarray,
                             labels: np.ndarray,
                             k: int,
                             ridge: float):
    """
    Fit diagonal Gaussian emission parameters for each state.
    """
    D = X.shape[1]
    means = np.zeros((k, D), dtype=float)
    variances = np.zeros((k, D), dtype=float)

    for c in range(k):
        Xi = X[labels == c]
        if Xi.size == 0:
            means[c] = 0.0
            variances[c] = 1.0
        else:
            means[c] = Xi.mean(axis=0)
            v = Xi.var(axis=0) + ridge
            variances[c] = np.maximum(v, ridge)

    return means, variances


def logpdf_diag_gauss(X: np.ndarray,
                      mean: np.ndarray,
                      var: np.ndarray) -> np.ndarray:
    """
    Log-density of diagonal Gaussian for each row of X.
    """
    D = X.shape[1]
    inv_var = 1.0 / var
    log_det = np.sum(np.log(var))
    diff = X - mean
    quad = np.sum(diff * diff * inv_var, axis=1)
    return -0.5 * (D * np.log(2 * np.pi) + log_det + quad)


def viterbi(log_pi0: np.ndarray,
            log_P: np.ndarray,
            logB: np.ndarray) -> np.ndarray:
    """
    Standard Viterbi algorithm in log-probability space.
    """
    T, k = logB.shape
    dp = np.empty((T, k), dtype=float)
    backp = np.empty((T, k), dtype=int)

    dp[0] = log_pi0 + logB[0]
    backp[0] = -1

    for t in range(1, T):
        for j in range(k):
            scores = dp[t - 1] + log_P[:, j]
            best_prev = int(np.argmax(scores))
            backp[t, j] = best_prev
            dp[t, j] = scores[best_prev] + logB[t, j]

    path = np.empty(T, dtype=int)
    path[-1] = int(np.argmax(dp[-1]))
    for t in range(T - 2, -1, -1):
        path[t] = backp[t + 1, path[t + 1]]

    return path


def hmm_smooth(df: pd.DataFrame,
               labels_lr: np.ndarray,
               Z_spec: np.ndarray,
               k: int):
    """
    Choose emission features, fit diagonal Gaussians, and run
    Viterbi per hole to obtain labels_hmm and P.
    """
    if CFG.use_Zspec_for_emissions:
        D = min(CFG.emission_dims, Z_spec.shape[1])
        X_emit = Z_spec[:, :D]
    else:
        X_emit = df[list(CFG.GEOCHEM_COLUMNS)].to_numpy(float)

    C = transition_counts(df, labels_lr, k)
    P = row_normalize_with_constraints(C)

    means, vars_ = gaussian_emission_params(X_emit, labels_lr, k, CFG.min_cov_ridge)

    # Initial state probabilities from first samples in each hole
    pi0 = np.ones(k, dtype=float)
    zname = CFG.xyz_cols[2]
    for _, g in df.groupby(CFG.id_col, sort=False):
        idx = g.index.to_numpy()
        order = np.argsort(g[zname].to_numpy())
        if len(order):
            first_state = labels_lr[idx[order[0]]]
            pi0[first_state] += 1.0
    pi0 /= pi0.sum()

    logP = np.log(np.maximum(P, 1e-16))
    log_pi0 = np.log(np.maximum(pi0, 1e-16))

    labels_out = labels_lr.copy()

    for _, g in df.groupby(CFG.id_col, sort=False):
        idx = g.index.to_numpy()
        order = np.argsort(g[zname].to_numpy())
        idx = idx[order]

        Xe = X_emit[idx]
        logB = np.column_stack(
            [logpdf_diag_gauss(Xe, means[c], vars_[c]) for c in range(k)]
        )
        path = viterbi(log_pi0, logP, logB)
        labels_out[idx] = path.astype(int)

    return labels_out, P


# -----------------------------------------------------------------------------
# LSTM sequence model and RNN smoothing
# -----------------------------------------------------------------------------

def build_sequences(df: pd.DataFrame,
                    labels: np.ndarray,
                    Z_spec: np.ndarray,
                    k: int,
                    S_mean: np.ndarray):
    """
    Build sliding depth windows and feature vectors, using labels as targets.

    Features per time step:
      - normalized coordinates (Z, X, Y)
      - first three spectral components
      - distances to spectral centroids
      - local continuity statistic (S_mean)
    """
    features: List[np.ndarray] = []
    targets: List[np.ndarray] = []

    centroids = np.stack(
        [Z_spec[labels == c].mean(axis=0) for c in range(k)],
        axis=0
    )

    zname = CFG.xyz_cols[2]

    for _, g in df.groupby(CFG.id_col, sort=False):
        idx = g.index.to_numpy()
        if len(idx) < CFG.seq_len:
            continue

        order = np.argsort(g[zname].to_numpy())
        idx = idx[order]

        X = g.loc[idx, CFG.xyz_cols[0]].to_numpy()
        Y = g.loc[idx, CFG.xyz_cols[1]].to_numpy()
        Z = g.loc[idx, CFG.xyz_cols[2]].to_numpy()

        def norm(v):
            vmin, vmax = v.min(), v.max()
            return (v - vmin) / (vmax - vmin + 1e-12)

        x_n, y_n, z_n = norm(X), norm(Y), norm(Z)

        Zloc = Z_spec[idx, :]
        d_cent = cdist(Zloc, centroids)
        Sloc = S_mean[idx][:, None]

        Xh = np.concatenate(
            [z_n[:, None], x_n[:, None], y_n[:, None],
             Zloc[:, :3], d_cent, Sloc],
            axis=1,
        )
        labs = labels[idx]

        for i in range(len(idx) - CFG.seq_len + 1):
            features.append(Xh[i:i + CFG.seq_len])
            targets.append(labs[i:i + CFG.seq_len])

    if not features:
        return np.zeros((0, CFG.seq_len, 4), dtype=np.float32), \
               np.zeros((0, CFG.seq_len), dtype=int)

    X_seq = np.array(features, dtype=np.float32)
    Y_seq = np.array(targets, dtype=int)
    return X_seq, Y_seq


def make_markov_loss(P: np.ndarray, lambda_markov: float):
    """
    Loss = cross-entropy + λ * Markov consistency term based on P.
    """
    P_tf = tf.constant(P, dtype=tf.float32)
    eps = tf.constant(1e-7, dtype=tf.float32)

    def loss_fn(y_true, y_pred):
        ce = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
        ce = tf.reduce_mean(ce)

        if lambda_markov <= 0.0:
            return ce

        p_t = y_pred[:, :-1, :]
        p_tp1 = y_pred[:, 1:, :]

        M = tf.einsum("btk,btj->kj", p_t, p_tp1)
        M = M / (tf.reduce_sum(M, axis=1, keepdims=True) + eps)

        kl = tf.reduce_sum(
            M * (tf.math.log(M + eps) - tf.math.log(P_tf + eps)),
            axis=1,
        )
        kl = tf.reduce_mean(kl)

        return ce + lambda_markov * kl

    return loss_fn


def lstm_refine(df: pd.DataFrame,
                Z_spec: np.ndarray,
                labels_in: np.ndarray,
                k: int,
                S_mean: np.ndarray,
                P: np.ndarray):
    """
    Train an LSTM sequence model and apply it with overlapping windows and
    majority voting to obtain RNN-smoothed labels labels_lstm.
    """
    if not (CFG.USE_LSTM and tf is not None):
        return labels_in

    X_seq, Y_seq = build_sequences(df, labels_in, Z_spec, k, S_mean)
    if X_seq.shape[0] == 0:
        return labels_in

    model = Sequential([
        LSTM(128, return_sequences=True,
             input_shape=(X_seq.shape[1], X_seq.shape[2])),
        LSTM(64, return_sequences=True),
        TimeDistributed(Dense(k, activation="softmax")),
    ])

    loss_fn = make_markov_loss(P, CFG.lambda_markov)
    model.compile(optimizer="adam", loss=loss_fn, metrics=["accuracy"])

    callbacks = [
        EarlyStopping(
            monitor="val_accuracy",
            patience=CFG.patience,
            restore_best_weights=True,
        )
    ]

    model.fit(
        X_seq,
        Y_seq,
        validation_split=CFG.val_split,
        epochs=CFG.epochs,
        batch_size=CFG.batch_size,
        callbacks=callbacks,
        verbose=0,
    )

    preds_all = labels_in.copy()
    zname = CFG.xyz_cols[2]

    centroids_global = np.stack(
        [Z_spec[labels_in == c].mean(axis=0) for c in range(k)],
        axis=0
    )

    for _, g in df.groupby(CFG.id_col, sort=False):
        idx = g.index.to_numpy()
        if len(idx) < CFG.seq_len:
            continue

        order = np.argsort(g[zname].to_numpy())
        idx = idx[order]

        X = g.loc[idx, CFG.xyz_cols[0]].to_numpy()
        Y = g.loc[idx, CFG.xyz_cols[1]].to_numpy()
        Z = g.loc[idx, CFG.xyz_cols[2]].to_numpy()

        def norm(v):
            vmin, vmax = v.min(), v.max()
            return (v - vmin) / (vmax - vmin + 1e-12)

        x_n, y_n, z_n = norm(X), norm(Y), norm(Z)

        Zloc = Z_spec[idx, :]
        d_cent = cdist(Zloc, centroids_global)
        Sloc = S_mean[idx][:, None]

        Xh = np.concatenate(
            [z_n[:, None], x_n[:, None], y_n[:, None],
             Zloc[:, :3], d_cent, Sloc],
            axis=1,
        )

        votes: Dict[int, List[int]] = {int(i): [] for i in idx}

        for i in range(len(idx) - CFG.seq_len + 1):
            Xin = Xh[i:i + CFG.seq_len][None, ...]
            Yhat = model.predict(Xin, verbose=0)[0]
            y_seq = np.argmax(Yhat, axis=1)
            for t, j in enumerate(range(i, i + CFG.seq_len)):
                votes[int(idx[j])].append(int(y_seq[t]))

        for j in idx:
            if votes[int(j)]:
                vals, counts = np.unique(votes[int(j)], return_counts=True)
                preds_all[j] = int(vals[np.argmax(counts)])

    return preds_all


# -----------------------------------------------------------------------------
# Post-processing: thickness rule + max contiguous units per hole
# -----------------------------------------------------------------------------

def _segments(labels: np.ndarray):
    """Helper: return (start, end, label) for each contiguous segment."""
    segments = []
    s = 0
    while s < len(labels):
        e = s
        while e + 1 < len(labels) and labels[e + 1] == labels[s]:
            e += 1
        segments.append((s, e, int(labels[s])))
        s = e + 1
    return segments


def enforce_min_thickness(df: pd.DataFrame,
                          labels: np.ndarray,
                          P: np.ndarray) -> np.ndarray:
    """
    Enforce a minimum thickness rule by merging intervals that are
    thinner than the chosen threshold, guided by P.
    """
    out = labels.copy()
    zname = CFG.xyz_cols[2]

    changed = True
    it = 0

    while changed and it < CFG.merge_max_iters:
        changed = False
        it += 1

        for _, g in df.groupby(CFG.id_col, sort=False):
            idx = g.index.to_numpy()
            order = np.argsort(g[zname].to_numpy())
            idx = idx[order]
            labs = out[idx].copy()

            if len(labs) == 0:
                continue

            threshold_samples = int(CFG.min_unit_samples)
            z = g.loc[idx, zname].to_numpy()
            dz = np.diff(z)
            dz = dz[np.isfinite(dz) & (dz > 0)]

            if dz.size and CFG.min_unit_m is not None:
                median_dz = float(np.median(dz))
                if median_dz > 0:
                    thr_m = int(np.ceil(CFG.min_unit_m / median_dz))
                    threshold_samples = max(threshold_samples, thr_m)

            threshold_samples = max(threshold_samples, 1)

            segments = _segments(labs)

            for s, e, lab in segments:
                length = e - s + 1
                if length >= threshold_samples:
                    continue

                left_state = labs[s - 1] if s > 0 else None
                right_state = labs[e + 1] if e + 1 < len(labs) else None

                if left_state is None and right_state is None:
                    continue

                if left_state is None:
                    labs[s:e + 1] = right_state
                elif right_state is None:
                    labs[s:e + 1] = left_state
                else:
                    prev_state = left_state
                    next_state = right_state
                    sl = (np.log(P[prev_state, left_state] + 1e-12) +
                          np.log(P[left_state, next_state] + 1e-12))
                    sr = (np.log(P[prev_state, right_state] + 1e-12) +
                          np.log(P[right_state, next_state] + 1e-12))
                    labs[s:e + 1] = left_state if sl >= sr else right_state

                changed = True

            out[idx] = labs

    return out


def enforce_max_units(df: pd.DataFrame,
                      labels: np.ndarray,
                      P: np.ndarray) -> np.ndarray:
    """
    Enforce a per-hole cap on the number of contiguous zones (≤ max_units_per_hole).
    """
    out = labels.copy()
    zname = CFG.xyz_cols[2]

    for _, g in df.groupby(CFG.id_col, sort=False):
        idx = g.index.to_numpy()
        order = np.argsort(g[zname].to_numpy())
        idx = idx[order]
        labs = out[idx].copy()

        it = 0
        while it < CFG.maxcap_max_iters:
            segments = _segments(labs)
            if len(segments) <= CFG.max_units_per_hole:
                break

            lengths = np.array([e - s + 1 for (s, e, _) in segments])
            j_min = int(np.argmin(lengths))
            s, e, lab = segments[j_min]

            left_state = labs[s - 1] if s > 0 else None
            right_state = labs[e + 1] if e + 1 < len(labs) else None

            if left_state is None and right_state is None:
                break

            if left_state is None:
                labs[s:e + 1] = right_state
            elif right_state is None:
                labs[s:e + 1] = left_state
            else:
                prev_state = left_state
                next_state = right_state
                sl = (np.log(P[prev_state, left_state] + 1e-12) +
                      np.log(P[left_state, next_state] + 1e-12))
                sr = (np.log(P[prev_state, right_state] + 1e-12) +
                      np.log(P[right_state, next_state] + 1e-12))
                labs[s:e + 1] = left_state if sl >= sr else right_state

            it += 1

        out[idx] = labs

    return out


# -----------------------------------------------------------------------------
# Main driver: follows the flowchart from "Joint Spatial Continuity" downward
# -----------------------------------------------------------------------------

def main():
    df = read_data(CFG.data_csv)

    X_geo = df[list(CFG.GEOCHEM_COLUMNS)].to_numpy(float)
    xyz = df[list(CFG.xyz_cols)].to_numpy(float)

    # Joint spatial continuity
    A, S_mean = build_affinity_matrices(X_geo, xyz)

    # Spectral embedding (Z_spec)
    Z_spec = spectral_embed_sparse(A, dim=CFG.spectral_dim)

    # First elbow test on Z_spec  → k
    k = elbow_k(Z_spec, CFG.kmin, CFG.kmax)

    k_prev = None
    labels_final = None
    labels0_save = labels_lr_save = labels_hmm_save = labels_lstm_save = None
    P_save = None

    # "Repeat Elbow (k*)" loop, as in the figure
    for loop_idx in range(CFG.max_reloops):
        if k_prev is not None and k == k_prev and labels_final is not None:
            break
        k_prev = k

        # k-means clustering (in embedded space)
        kmeans = KMeans(n_clusters=k, n_init=20, random_state=CFG.seed)
        labels_0 = kmeans.fit_predict(Z_spec)

        # Left-to-right ordering by median depth
        labels_lr = relabel_by_depth(df, labels_0)

        # Markov chains + HMM smoothing
        labels_hmm, P = hmm_smooth(df, labels_lr, Z_spec, k)

        # LSTM sequence model (RNN smoothing)
        labels_lstm = lstm_refine(df, Z_spec, labels_hmm, k, S_mean, P)

        labels0_save = labels_0
        labels_lr_save = labels_lr
        labels_hmm_save = labels_hmm
        labels_lstm_save = labels_lstm
        P_save = P
        labels_final = labels_lstm

        # Repeat elbow on Z_spec (conceptual k*)
        k_star = elbow_k(Z_spec, CFG.kmin, CFG.kmax)
        if k_star == k:
            break
        k = k_star

    if labels_final is None:
        raise RuntimeError("GkRNN did not produce any labels.")

    # Final domaining and zoning of the tailings deposit
    labels_thick = enforce_min_thickness(df, labels_final, P_save)
    labels_final = enforce_max_units(df, labels_thick, P_save)

    k_global = int(k)

    df_out = df.copy()
    df_out["Cluster_kmeans"] = labels0_save.astype(int)
    df_out["Cluster_lr"] = labels_lr_save.astype(int)
    df_out["Zone_HMM"] = labels_hmm_save.astype(int)
    df_out["Zone_LSTM"] = labels_lstm_save.astype(int)
    df_out["Zone_Final"] = labels_final.astype(int)
    df_out["k_global"] = k_global

    df_out.to_csv(CFG.out_csv, index=False)
    pd.DataFrame(P_save).to_csv("gkrnn_transition_matrix_P.csv", index=False)

    print(f"GkRNN completed. Saved zones to '{CFG.out_csv}' (k = {k_global}).")


if __name__ == "__main__":
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        main()